In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

In [ ]:
###### REQUIRES DHS_Spatial_Merged.csv FROM ERA_DHS_EXPOSURE_VARIABLE_BUILDING.ipynb


# ==========================================
# 0. CONFIGURATION & REGIONAL MAPPING
# ==========================================
REGION_MAPPING = {
    # West Africa
    'Benin': 'West Africa', 'Burkina Faso': 'West Africa', 'Cote dIvoire': 'West Africa',
    'Gambia': 'West Africa', 'Ghana': 'West Africa', 'Liberia': 'West Africa',
    'Mali': 'West Africa', 'Mauritania': 'West Africa', 'Nigeria': 'West Africa',
    'Senegal': 'West Africa', 'Sierra Leone': 'West Africa', 'Togo': 'West Africa',
    # Central Africa
    'Angola': 'Central Africa', 'Cameroon': 'Central Africa', 'Chad': 'Central Africa',
    'DR Congo': 'Central Africa', 'Gabon': 'Central Africa',
    # East Africa
    'Burundi': 'East Africa', 'Comoros': 'East Africa', 'Ethiopia': 'East Africa',
    'Kenya': 'East Africa', 'Madagascar': 'East Africa', 'Malawi': 'East Africa',
    'Mozambique': 'East Africa', 'Rwanda': 'East Africa', 'Tanzania': 'East Africa',
    'Uganda': 'East Africa', 'Zambia': 'East Africa', 'Zimbabwe': 'East Africa',
    # Southern Africa
    'Lesotho': 'Southern Africa', 'Namibia': 'Southern Africa', 'South Africa': 'Southern Africa'
}

# EXPANDED SHOCK SPECS: Updated to 90-95 for Precip, and 90-100 for Soil Moisture
SHOCK_SPECS = {
    'Precipitation (95-100th Pct)': {
        'type': 'precip', 'col': 'exceed_95_100', 'title': 'Table 1: Extreme Precipitation (95-100th Percentile)'
    },
    'Precipitation (90-95th Pct)': {
        'type': 'precip', 'col': 'exceed_90_95', 'title': 'Table 2: Extreme Precipitation (90-95th Percentile)'
    },
    'Soil Moisture: Wet in Wet (95-100th Pct)': {
        'type': 'soil', 'col': 'Wet_in_Wet_95_100', 'title': 'Table 3: Soil Moisture - Wet in Wet (95-100th Percentile)'
    },
    'Soil Moisture: Wet in Wet (90-100th Pct)': {
        'type': 'soil', 'col': 'Wet_in_Wet_90_100', 'title': 'Table 4: Soil Moisture - Wet in Wet (90-100th Percentile)'
    },
    'Soil Moisture: Wet in Dry (95-100th Pct)': {
        'type': 'soil', 'col': 'Wet_in_Dry_95_100', 'title': 'Table 5: Soil Moisture - Wet in Dry (95-100th Percentile)'
    },
    'Soil Moisture: Wet in Dry (90-100th Pct)': {
        'type': 'soil', 'col': 'Wet_in_Dry_90_100', 'title': 'Table 6: Soil Moisture - Wet in Dry (90-100th Percentile)'
    },
    'Soil Moisture: Dry in Wet (0-5th Pct)': {
        'type': 'soil', 'col': 'Dry_in_Wet_0_5', 'title': 'Table 7: Soil Moisture - Dry in Wet (0-5th Percentile)'
    },
    'Soil Moisture: Dry in Wet (0-10th Pct)': {
        'type': 'soil', 'col': 'Dry_in_Wet_0_10', 'title': 'Table 8: Soil Moisture - Dry in Wet (0-10th Percentile)'
    },
    'Soil Moisture: Dry in Dry (0-5th Pct)': {
        'type': 'soil', 'col': 'Dry_in_Dry_0_5', 'title': 'Table 9: Soil Moisture - Dry in Dry (0-5th Percentile)'
    },
    'Soil Moisture: Dry in Dry (0-10th Pct)': {
        'type': 'soil', 'col': 'Dry_in_Dry_0_10', 'title': 'Table 10: Soil Moisture - Dry in Dry (0-10th Percentile)'
    }
}

TARGET_MAP = {
    'sought_any': 'Disclosed Violence (Any)',
    'informal': 'Informal Help',
    'formal': 'Formal Help'
}

demographics_cols = ['v012', 'v025', 'v106', 'v190', 'v000', 'v024']
demographics_formula = "v012 + C(v025) + C(v106) + C(v190) + C(v000) + C(v024)"
scaler = StandardScaler()

# ==========================================
# 1. BATCH COUNTRY MODEL ESTIMATION
# ==========================================
summary_records = []
processed_files = glob.glob('/content/Processed_Country_Data/*_DHS_Spatial_Merged.csv')

print(f"Running threshold estimates across {len(processed_files)} country datasets...")

for f in processed_files:
    country_name = os.path.basename(f).replace('_DHS_Spatial_Merged.csv', '')
    region = REGION_MAPPING.get(country_name, 'Other')

    try:
        df_c = pd.read_csv(f)
    except Exception as e:
        print(f"Error reading {country_name} CSV: {e}")
        continue

    df_c['weight'] = df_c['d005'] / 1e6

    # Run regressions for each specified climate shock
    for shock_label, spec in SHOCK_SPECS.items():
        shock_col = spec['col']

        if shock_col not in df_c.columns:
            print(f"[{country_name}] Missing column: '{shock_col}'. Skipping...")
            continue

        # Drop NaNs just for scaling to avoid errors
        df_scaling = df_c.dropna(subset=[shock_col])
        if df_scaling.empty:
            continue

        df_c[f'z_{shock_col}'] = np.nan
        df_c.loc[df_scaling.index, f'z_{shock_col}'] = scaler.fit_transform(df_scaling[[shock_col]])

        for v_col, v_title in TARGET_MAP.items():
            df_clean = df_c.dropna(subset=[v_col, f'z_{shock_col}'] + demographics_cols + ['weight'])
            if len(df_clean) < 50:
                continue

            formula = f"{v_col} ~ z_{shock_col} + {demographics_formula}"
            try:
                group_codes = pd.factorize(df_clean['v024'])[0]
                model = smf.glm(formula=formula, data=df_clean, family=sm.families.Binomial(), freq_weights=df_clean['weight'])
                res = model.fit(cov_type='cluster', cov_kwds={'groups': group_codes})

                coef = res.params[f'z_{shock_col}']
                bse = res.bse[f'z_{shock_col}']
                pval = res.pvalues[f'z_{shock_col}']

                # --- SAFEGUARD FOR MAURITANIA / COMOROS EDGE CASES ---
                if np.isnan(coef) or np.isnan(bse) or bse > 10:
                    continue
                # ---------------------------------------------------

                pct_change = (np.exp(coef) - 1) * 100
                ci_low = (np.exp(coef - 1.96 * bse) - 1) * 100
                ci_high = (np.exp(coef + 1.96 * bse) - 1) * 100
                is_sig = pval < 0.05

                cell_str = f"**{pct_change:+.1f}% [{ci_low:+.1f}%, {ci_high:+.1f}%]**" if is_sig else f"{pct_change:+.1f}% [{ci_low:+.1f}%, {ci_high:+.1f}%]"

                summary_records.append({
                    'Country': country_name,
                    'Region': region,
                    'Shock': shock_label,
                    'Outcome': v_title,
                    'Pct_Change': pct_change,
                    'CI_Low': ci_low,
                    'CI_High': ci_high,
                    'Significant': is_sig,
                    'Formatted_Result': cell_str
                })
            except Exception as e:
                print(f"[{country_name}] Regression error for {shock_label} ({v_col}): {e}")

df_all_results = pd.DataFrame(summary_records)

# ==========================================
# 2. GENERATE NATURE-READY TABLES
# ==========================================
if df_all_results.empty:
    print("\n" + "!"*80)
    print("CRITICAL ERROR: No valid regression results were generated.")
    print("Look at the diagnostic prints above to see why columns were skipped or models failed.")
    print("!"*80)
else:
    region_order = ['West Africa', 'Central Africa', 'East Africa', 'Southern Africa']

    print("\n" + "="*80)
    print(" NATURE-STYLE TABLES: DOSE-RESPONSE & 95% CONFIDENCE INTERVALS")
    print("="*80)

    for shock_label, spec in SHOCK_SPECS.items():
        df_shock = df_all_results[df_all_results['Shock'] == shock_label]
        if df_shock.empty:
            continue

        pivot_table = df_shock.pivot(index=['Region', 'Country'], columns='Outcome', values='Formatted_Result').reset_index()

        pivot_table['Region'] = pd.Categorical(pivot_table['Region'], categories=region_order, ordered=True)
        pivot_table['Region'] = pivot_table['Region'].cat.add_categories('N/A')
        pivot_table = pivot_table.sort_values(['Region', 'Country']).fillna("N/A")

        cols = ['Region', 'Country', 'Disclosed Violence (Any)', 'Informal Help', 'Formal Help']
        pivot_table = pivot_table[[c for c in cols if c in pivot_table.columns]]

        print(f"\n### {spec['title']}")
        print(pivot_table.to_markdown(index=False))

        out_filename = shock_label.replace(' ', '_').replace(':', '').replace('>', 'gt').replace('<', 'lt').replace('(', '').replace(')', '')
        pivot_table.to_csv(f'/content/Table_{out_filename}.csv', index=False)

    # ==========================================
    # 3. NATURE-READY FOREST PLOTS
    # ==========================================
    plt.rcParams['font.sans-serif'] = 'Helvetica'
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['axes.edgecolor'] = '#333333'
    plt.rcParams['axes.linewidth'] = 0.8

    for shock_label, spec in SHOCK_SPECS.items():
        df_shock = df_all_results[df_all_results['Shock'] == shock_label].copy()
        if df_shock.empty:
            continue

        df_shock['Region'] = pd.Categorical(df_shock['Region'], categories=region_order, ordered=True)
        df_shock = df_shock.sort_values(['Region', 'Country'], ascending=[True, False])

        fig, axes = plt.subplots(1, 3, figsize=(16, max(6, len(df_shock['Country'].unique()) * 0.35)), sharey=True)
        fig.suptitle(f"{shock_label}\n(Dose-Response % Odds Change & 95% CIs)", fontsize=14, fontweight='bold', y=0.98)

        for idx, (v_col, v_title) in enumerate(TARGET_MAP.items()):
            ax = axes[idx]
            sub = df_shock[df_shock['Outcome'] == v_title].copy()

            if sub.empty:
                continue

            y_positions = np.arange(len(sub))
            colors = ['#d62728' if sig else '#4c4c4c' for sig in sub['Significant']]

            ax.errorbar(
                sub['Pct_Change'], y_positions,
                xerr=[sub['Pct_Change'] - sub['CI_Low'], sub['CI_High'] - sub['Pct_Change']],
                fmt='none', ecolor='#4c4c4c', elinewidth=1.2, capsize=2.5, zorder=2
            )
            ax.scatter(sub['Pct_Change'], y_positions, c=colors, s=40, zorder=3, edgecolor='white', linewidth=0.5)

            ax.axvline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.7, zorder=1)
            ax.set_yticks(y_positions)
            ax.set_yticklabels(sub['Country'], fontsize=10)
            ax.set_title(v_title, fontsize=12, fontweight='bold', pad=10)
            ax.set_xlabel('% Change in Odds', fontsize=10)
            ax.grid(True, linestyle=':', alpha=0.5, axis='x')

            # --- LIMIT X-AXIS RANGE TO [-50%, +50%] ---
            ax.set_xlim(-50, 50)

        sns.despine(top=True, right=True)
        plt.tight_layout()

        fig_filename = shock_label.replace(' ', '_').replace(':', '').replace('>', 'gt').replace('<', 'lt').replace('(', '').replace(')', '')
        plt.savefig(f'/content/Nature_ForestPlot_{fig_filename}.png', dpi=300, bbox_inches='tight')
        plt.show()

    print("\nAll Nature-ready tables and forest plots successfully generated and saved to /content/")